# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset DOI**: [10.71728/senscience.vq4a-28xa](https://doi.org/10.71728/senscience.vq4a-28xa)

(c) Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Display metadata (accessing properties, do NOT treat metadata as dict)
md = dataset.metadata
print(f"Dataset: {md.name}")
print(f"Description: {md.description}\n")
print(f"Version: {md.version}")
print(f"License: {md.license}")
print(f"Cite as: {md.citeAs}")
print(f"Published: {md.datePublished}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all record sets and their `@id`s, then show their fields and field `@id`s. When using `mlcroissant`, you should reference record sets and fields by their `@id`.


In [ ]:
# Helper to pretty print record set and field structure
def explore_record_sets(ds):
    print("Available record sets and fields:")
    for rset in ds.metadata.recordSets:
        print(f"- RecordSet name: {rset.name}, @id: '{rset.id}'")
        if hasattr(rset, 'fields') and rset.fields:
            for f in rset.fields:
                print(f"    - Field: {f.name if hasattr(f, 'name') else ''}, @id: '{f.id}' (dataType: {getattr(f, 'dataType', '?')})")

# Do the exploration
explore_record_sets(dataset)

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. All entities (record sets, fields, columns, etc.) must be referenced by their `@id` as specified in the Croissant schema.

**Tip:** Use `explore_record_sets` output above to choose record set `@id`s.


In [ ]:
# Replace with actual record set @ids (found above)
# For this dataset, let us discover all recordSet @ids:
record_sets = [rset.id for rset in dataset.metadata.recordSets]

dataframes = {}
for record_set_id in record_sets:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"@id: {record_set_id}\nShape: {df.shape}\nColumns: {df.columns.tolist()}\n")
    except Exception as ex:
        print(f"Error extracting '{record_set_id}':", ex)

# Let's pick the first available record set for further exploration
# If no record sets found, report
if record_sets:
    example_rs_id = record_sets[0]
    print(f"Showing head for record set: {example_rs_id}")
    print(dataframes[example_rs_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, transforming, and grouping data for further insights.

**All field names used are their `@id`s as reported above.**

In [ ]:
# Analytic example: pick a numeric field from the above record set for demonstration
// Sample code will use placeholder field names that must be replaced with actual @id for your dataset
# For this dataset, let us enumerate available numeric columns:
import numpy as np

df = dataframes.get(example_rs_id)
numeric_candidates = []
if df is not None and not df.empty:
    for col in df.columns:
        try:
            # Attempt to coerce to numeric
            vals = pd.to_numeric(df[col], errors='coerce')
            # If there are numeric values, consider as candidate
            if np.isfinite(vals).sum() > 0:
                numeric_candidates.append(col)
        except Exception:
            continue

    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]  # Use the first numeric field
        print(f"Analyzing numeric field: {numeric_field}")
        # Coerce numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as cutoff
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field, e.g., the first non-numeric column
        group_field = None
        for col in df.columns:
            if col not in numeric_candidates:
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No data available in selected record set.")

## 5. Visualization
Visualize distributions or relationships for the main numeric field in the dataset. All field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and len(numeric_candidates) > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If a group field was set above
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore datasets described with the Croissant standard using the `mlcroissant` library. 

- All references to record sets, fields, and columns strictly use their Croissant `@id`.
- You can adapt steps above (section 4/5) to perform deeper or more specific analyses based on dataset structure.
- This approach ensures robust referencing and easy adaptation to schema changes.